# Master Dataset Prep — Before Modeling

**Continuation of Week 3, before moving into Week 4 (Modeling).**

The master table from the previous notebook (149 columns) isn't quite ready to be used directly for modeling. Three things get sorted out here:
1. Check the full correlation with `TARGET` — across all features, including the aggregated results from the 5 secondary tables
2. Check & handle multicollinearity between features (especially important for Logistic Regression)
3. Stratified train/test split, then save the final feature set


## 1. Load Master Table


In [1]:
import pandas as pd
import numpy as np

df_master = pd.read_csv('../data/processed/master_features.csv')
print(df_master.shape)

(307511, 149)


## 2. Full Correlation with Target

We now have the complete picture: original `application_train` features + aggregated features from `bureau`, `previous_application`, `installments`, `POS_CASH`, and `credit_card`, all at once.


In [2]:
correlations = df_master.select_dtypes(include=[np.number]).corr()['TARGET'].sort_values()

print("TOP 15 NEGATIVE (higher value = SAFER):")
print(correlations.head(15))

print("\nTOP 15 POSITIVE (higher value = RISKIER):")
print(correlations.tail(16))

TOP 15 NEGATIVE (higher value = SAFER):
EXT_SOURCE_2                 -0.160295
EXT_SOURCE_3                 -0.155892
EXT_SOURCE_1                 -0.098887
AGE_YEARS                    -0.078239
EMPLOYED_YEARS               -0.063368
PREV_RATIO_APPROVED          -0.041736
HAS_PROPERTY_INFO            -0.041392
AMT_GOODS_PRICE              -0.039623
BUREAU_COUNT_CLOSED          -0.037233
REGION_POPULATION_RELATIVE   -0.037227
AMT_CREDIT                   -0.030369
POS_COUNT                    -0.029678
FLAG_DOCUMENT_6              -0.028602
PREV_MEAN_AMT_ANNUITY        -0.026242
HOUR_APPR_PROCESS_START      -0.024166
Name: TARGET, dtype: float64

TOP 15 POSITIVE (higher value = RISKIER):
REG_CITY_NOT_WORK_CITY                0.050994
DAYS_ID_PUBLISH                       0.051457
DAYS_LAST_PHONE_CHANGE                0.055218
REGION_RATING_CLIENT                  0.058899
REGION_RATING_CLIENT_W_CITY           0.060893
ANOMALY_SCORE                         0.061664
INST_RATIO_SHORTFALL 

**Insight:**
- `EXT_SOURCE_2/3` remain the strongest predictors overall, but the week 3 engineered features are also on the radar: `PREV_RATIO_REFUSED` (0.078), `BUREAU_MEAN_DAYS_CREDIT` (0.084), `INST_RATIO_LATE` (0.071).
- **Important note:** `CC_MEAN_UTILIZATION` here only has a correlation of 0.065 — down significantly from the 0.136 seen in the feature engineering notebook. Reason: back then the correlation was computed only on customers who had credit card history, whereas in the master table customers without history are now filled with 0 (`finalize_master_features`). This "dilutes" the signal since most customers don't have a credit card. Worth noting as a limitation of this missing-value treatment.
- `FLAG_EMPLOYED_LONGER_THAN_POSSIBLE` shows up as `NaN` — consistent with the finding in notebook 2 (this feature has 0% variance, making the correlation undefined). So it's a definite candidate to drop.


## 3. Check Multicollinearity

With 149 columns, check pairs of features whose (absolute) correlation with each other is above 0.9.


In [3]:
feature_cols = [c for c in df_master.select_dtypes(include=[np.number]).columns 
                 if c not in ['TARGET', 'SK_ID_CURR']]

corr_matrix = df_master[feature_cols].corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(col, row, upper.loc[row, col]) 
                    for col in upper.columns 
                    for row in upper.index 
                    if upper.loc[row, col] > 0.9]

high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature_1', 'Feature_2', 'Correlation'])
high_corr_df = high_corr_df.sort_values('Correlation', ascending=False)
print(high_corr_df.shape)
print(high_corr_df.head(30))

(14, 3)
                      Feature_1                  Feature_2  Correlation
3                     AGE_YEARS                 DAYS_BIRTH     1.000000
4                EMPLOYED_YEARS              DAYS_EMPLOYED     1.000000
2      OBS_60_CNT_SOCIAL_CIRCLE   OBS_30_CNT_SOCIAL_CIRCLE     0.998491
0               AMT_GOODS_PRICE                 AMT_CREDIT     0.986734
9         INST_MEAN_AMT_PAYMENT   INST_MEAN_AMT_INSTALMENT     0.979679
6          PREV_MEAN_AMT_CREDIT  PREV_MEAN_AMT_APPLICATION     0.977106
11           POS_MAX_SK_DPD_DEF        POS_MEAN_SK_DPD_DEF     0.964838
12                CC_MAX_SK_DPD             CC_MEAN_SK_DPD     0.960642
1   REGION_RATING_CLIENT_W_CITY       REGION_RATING_CLIENT     0.950842
13                 CC_RATIO_DPD            CC_SUM_FLAG_DPD     0.944662
5           BUREAU_COUNT_CLOSED         BUREAU_COUNT_LOANS     0.932987
10               POS_MAX_SK_DPD            POS_MEAN_SK_DPD     0.924367
7       PREV_MEAN_DAYS_LAST_DUE   PREV_MEAN_DAYS_FIRST_D

**Insight:** there are 14 pairs of features that are nearly identical (correlation > 0.9). Some are automatically identical mathematically (`AGE_YEARS` vs `DAYS_BIRTH`, `EMPLOYED_YEARS` vs `DAYS_EMPLOYED` — correlation = 1.0, since one is just a linear transformation of the other). The rest are raw amount/count pairs that are naturally correlated (e.g. `AMT_GOODS_PRICE` vs `AMT_CREDIT`, `PREV_MEAN_AMT_CREDIT` vs `PREV_MEAN_AMT_APPLICATION`). One from each pair needs to be dropped so Model A (Logistic Regression) doesn't run into multicollinearity issues.


## 4. Train/Test Split (Stratified)

A stratified split is important because the target is imbalanced (~8% default) — this ensures the proportion stays consistent in both train and test.


In [4]:
from sklearn.model_selection import train_test_split

X = df_master.drop(columns=['TARGET'])
y = df_master['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train TARGET rate:", y_train.mean())
print("Test TARGET rate:", y_test.mean())

Train shape: (246008, 148)
Test shape: (61503, 148)
Train TARGET rate: 0.08072908198107379
Test TARGET rate: 0.08072776937710356


**Insight:** the target rate in train (8.073%) and test (8.073%) are nearly identical — stratification worked, so the model evaluation later won't be biased by a difference in target distribution.


## 5. Drop Redundant Features

Based on the findings in sections 2 & 3: drop 1 zero-variance feature + one from each of the 14 high-correlation pairs. Selection criteria: keep the version that's more interpretable or more relevant to risk (e.g. `AGE_YEARS` is kept over `DAYS_BIRTH`, `CC_RATIO_DPD` over `CC_SUM_FLAG_DPD` — consistent with the "ratio > raw count" finding from notebook 3).


In [5]:
# Columns to drop: FLAG_EMPLOYED_LONGER_THAN_POSSIBLE (zero variance)
# + one from each high-correlation pair

cols_to_drop = [
    'FLAG_EMPLOYED_LONGER_THAN_POSSIBLE',  # zero variance
    
    'DAYS_BIRTH',                    # keep AGE_YEARS (more interpretable)
    'DAYS_EMPLOYED',                 # keep EMPLOYED_YEARS
    'OBS_60_CNT_SOCIAL_CIRCLE',      # keep OBS_30 (shorter/more relevant window)
    'AMT_GOODS_PRICE',               # keep AMT_CREDIT (more relevant to risk)
    'INST_MEAN_AMT_PAYMENT',         # keep INST_MEAN_AMT_INSTALMENT
    'PREV_MEAN_AMT_APPLICATION',     # keep PREV_MEAN_AMT_CREDIT
    'POS_MEAN_SK_DPD_DEF',           # keep POS_MAX_SK_DPD_DEF (max captures risk more sharply)
    'CC_MEAN_SK_DPD',                # keep CC_MAX_SK_DPD
    'REGION_RATING_CLIENT',          # keep _W_CITY (more granular)
    'CC_SUM_FLAG_DPD',               # keep CC_RATIO_DPD (ratio > raw count, per our pattern)
    'BUREAU_COUNT_CLOSED',           # keep BUREAU_COUNT_LOANS
    'POS_MEAN_SK_DPD',               # keep POS_MAX_SK_DPD
    'PREV_MEAN_DAYS_FIRST_DUE',      # keep PREV_MEAN_DAYS_LAST_DUE
    'INST_SUM_FLAG_LATE',            # keep INST_SUM_FLAG_SHORTFALL
]

X_train_clean = X_train.drop(columns=cols_to_drop)
X_test_clean = X_test.drop(columns=cols_to_drop)

print("Before:", X_train.shape)
print("After:", X_train_clean.shape)

Before: (246008, 148)
After: (246008, 133)


## 6. Separate ID & Finalize Features

`SK_ID_CURR` is kept separately (for tracking/reference), not included as a model feature.


In [6]:
# Save SK_ID_CURR separately first (for reference/tracking, not for model features)
train_ids = X_train_clean['SK_ID_CURR']
test_ids = X_test_clean['SK_ID_CURR']

X_train_final = X_train_clean.drop(columns=['SK_ID_CURR'])
X_test_final = X_test_clean.drop(columns=['SK_ID_CURR'])

print(X_train_final.shape)
print(X_test_final.shape)

(246008, 132)
(61503, 132)


## 7. Save Final Dataset


In [7]:
X_train_final.to_csv('../data/processed/X_train.csv', index=False)
X_test_final.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
train_ids.to_csv('../data/processed/train_ids.csv', index=False)
test_ids.to_csv('../data/processed/test_ids.csv', index=False)

print("Saved!")

Saved!


---
### Summary
- The master table (149 columns) has had its full correlation and multicollinearity checked, and been split with stratification.
- 15 redundant features (1 zero-variance + 14 duplicate/high-correlation) were dropped.
- Final: **132 clean features**, saved as `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, plus `train_ids.csv`/`test_ids.csv` for tracking.
